In [15]:
import pandas as pd
import numpy as np
import sys   
from pathlib import Path
sys.path.append(str(Path.cwd().parent))   # add project root so src is importable
import matplotlib.pyplot as plt   
import seaborn as sns 

from src.config import (
    CUSTOMERS_TRAIN, LOANS_CLEAN, TRANSACTIONS_CLEAN, TRAIN_IDS, RANDOM_STATE,
)

train_cust = pd.read_parquet(CUSTOMERS_TRAIN)
loans = pd.read_parquet(LOANS_CLEAN)
txns  = pd.read_parquet(TRANSACTIONS_CLEAN)

from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize 

from scipy.stats import chi2_contingency

train_ids   = pd.read_parquet(TRAIN_IDS)["customer_id"]
loans_train = loans[loans["customer_id"].isin(train_ids)]
txns_train  = txns[txns["customer_id"].isin(train_ids)] 

In [16]:
loans_train1 = loans[loans["customer_id"].isin(train_ids)]
loans_train1= loans_train1.merge(train_cust, on="customer_id", how="left")

In [17]:
AVAILABLE = (loans_train1["inflow_to_loan_ratio"] > 1.2).sum()
AVAILABLE

np.int64(2597)

In [18]:
BASELINE= 0.24
TARGET = 0.16
ALPHA= 0.05
POWER= 0.80
AVAILABLE= 2597 
ARM= 2

effect=abs(proportion_effectsize(TARGET,BASELINE))

required= NormalIndPower().solve_power(
    effect_size=effect,
    alpha=ALPHA,
    power=POWER,
    ratio=1.0,
    alternative="two-sided",
)

have = AVAILABLE // ARM

print(f"MDE           : {BASELINE:.0%} -> {TARGET:.0%}   ({(TARGET-BASELINE)*100:.0f} points)")
print(f"effect size h : {effect:.4f}")
print(f"eligible      : {AVAILABLE} loans above the 1.2 cap") 
print(f"NEED per arm  : {required:.0f}")
print(f"HAVE per arm  : {have}")
print(f"VERDICT       : {'adequately powered' if have >= required else 'UNDERPOWERED - redesign'}")

MDE           : 24% -> 16%   (-8 points)
effect size h : 0.2009
eligible      : 2597 loans above the 1.2 cap
NEED per arm  : 389
HAVE per arm  : 1298
VERDICT       : adequately powered


In [19]:
# rng = np.random.default_rng(RANDOM_STATE)

# eligible = loans_train1[loans_train1["inflow_to_loan_ratio"] > 1.2].copy()

# # randomise CUSTOMERS, then let their loans inherit the arm
# cust = eligible["customer_id"].unique()
# assignment = pd.Series(
#     rng.choice(["control", "treatment"], size=len(cust)),
#     index=cust,
# )
# eligible["arm"] = eligible["customer_id"].map(assignment)

# print(f"eligible loans     : {len(eligible)}")
# print(f"eligible customers : {len(cust)}")
# print(eligible["arm"].value_counts(), "\n")


# # ── randomisation check: are the arms comparable? ─────────────
# numeric = ["credit_score", "wallet_tenure_months",
#            "avg_monthly_inflow_pkr", "inflow_to_loan_ratio", "amount_pkr"]
# print(eligible.groupby("arm")[numeric].median().round(1), "\n")

# for col in ["region", "purpose", "segment_true"]:
#     print(f"--- {col} ---")
#     print(pd.crosstab(eligible["arm"], eligible[col], normalize="index").round(3), "\n")

# # baseline default should also match — nothing has been applied yet
# print(eligible.groupby("arm")["defaulted"].agg(["mean", "size"]).round(4))

In [ ]:
# rng = np.random.default_rng(RANDOM_STATE)

# eligible = loans_train1[loans_train1["inflow_to_loan_ratio"] > 1.5].copy()

# # ── build the strata ──────────────────────────────────────────
# eligible["ratio_band"] = pd.qcut(eligible["inflow_to_loan_ratio"], 4)
# eligible["stratum"] = (eligible["ratio_band"].astype(str)
#                        + " | " + eligible["region"].astype(str))

# print("strata:", eligible["stratum"].nunique())
# print(eligible["stratum"].value_counts().tail(8))   # watch the small ones


# # ── randomise WITHIN each stratum ─────────────────────────────
# # shuffle everything once, then alternate control/treatment inside
# # each stratum so every stratum splits ~50/50 by construction
# eligible = eligible.sample(frac=1, random_state=RANDOM_STATE)

# eligible["arm"] = np.where(
#     eligible.groupby("stratum", observed=True).cumcount() % 2 == 0,
#     "control", "treatment"
# )

# print(eligible["arm"].value_counts(), "\n")


# # ── same balance check as before ──────────────────────────────
# numeric = ["credit_score", "wallet_tenure_months",
#            "avg_monthly_inflow_pkr", "inflow_to_loan_ratio", "amount_pkr"]
# print(eligible.groupby("arm")[numeric].median().round(1), "\n")

# for col in ["region", "purpose", "segment_true"]:
#     print(f"--- {col} ---")
#     print(pd.crosstab(eligible["arm"], eligible[col], normalize="index").round(3), "\n")

# print(eligible.groupby("arm")["defaulted"].agg(["mean", "size"]).round(4))

strata: 24
stratum
(2.31, 4.02] | AJK-GB                       36
(7.792, 49.05] | Balochistan                35
(1.5090000000000001, 2.31] | AJK-GB         30
(1.5090000000000001, 2.31] | Balochistan    26
(7.792, 49.05] | AJK-GB                     25
(2.31, 4.02] | Balochistan                  24
(4.02, 7.792] | AJK-GB                      23
(4.02, 7.792] | Balochistan                 20
Name: count, dtype: int64
arm
control      1146
treatment    1140
Name: count, dtype: int64 

           credit_score  wallet_tenure_months  avg_monthly_inflow_pkr  \
arm                                                                     
control           467.0                  24.0                 26500.0   
treatment         462.0                  23.0                 26400.0   

           inflow_to_loan_ratio  amount_pkr  
arm                                          
control                     4.0    111559.5  
treatment                   4.0    115730.0   

--- region ---
region     AJK-GB

In [ ]:
# # ══════════════════════════════════════════════════════════════
# # STEP 6 — apply the cap, then simulate outcomes
# # ══════════════════════════════════════════════════════════════
# CAP = 1.5
# TRUE_CONTROL   = 0.236      # observed baseline in both arms
# TRUE_TREATMENT = 0.156      # the effect we INJECT: an 8-point reduction

# rng = np.random.default_rng(RANDOM_STATE)

# # --- the intervention: treatment loans get capped -------------
# eligible["capped_amount"] = np.where(
#     eligible["arm"] == "treatment",
#     eligible["avg_monthly_inflow_pkr"] * CAP,     # shrunk to the cap
#     eligible["amount_pkr"],                       # control untouched
# )

# # --- GUARDRAIL: what did that cost in volume? -----------------
# vol = eligible.groupby("arm")["capped_amount"].sum()
# drop = (vol["control"] - vol["treatment"]) / vol["control"]
# print(f"disbursed  control   {vol['control']:>15,.0f}")
# print(f"disbursed  treatment {vol['treatment']:>15,.0f}")
# print(f"VOLUME DROP: {drop:.1%}   (guardrail: 15% max)")
# print("GUARDRAIL:", "PASS" if drop <= 0.15 else "BREACHED", "\n")

# # --- simulate the outcome -------------------------------------
# p = np.where(eligible["arm"] == "treatment", TRUE_TREATMENT, TRUE_CONTROL)
# eligible["outcome"] = rng.random(len(eligible)) < p


# # ══════════════════════════════════════════════════════════════
# # STEP 7 — effect size
# # ══════════════════════════════════════════════════════════════
# res = eligible.groupby("arm")["outcome"].agg(["mean", "size"])
# print(res.round(4))

# gap = res.loc["control", "mean"] - res.loc["treatment", "mean"]
# print(f"\nEFFECT: {gap*100:.2f} point reduction in default\n")


# # ══════════════════════════════════════════════════════════════
# # STEP 8 — chi-square
# # ══════════════════════════════════════════════════════════════
# table = pd.crosstab(eligible["arm"], eligible["outcome"])
# print(table)

# chi2, pval, dof, expected = chi2_contingency(table)
# print(f"\np = {pval:.6f}")
# print("VERDICT:", "significant" if pval < 0.05 else "not significant")


# # ══════════════════════════════════════════════════════════════
# # STEP 9 — validity check: did the analysis recover what we injected?
# # ══════════════════════════════════════════════════════════════
# injected = (TRUE_CONTROL - TRUE_TREATMENT) * 100
# print(f"\ninjected effect : {injected:.1f} points")
# print(f"measured effect : {gap*100:.2f} points")
# print(f"difference      : {abs(gap*100 - injected):.2f} points  (sampling noise)")

disbursed  control       178,747,219
disbursed  treatment      54,392,250
VOLUME DROP: 69.6%   (guardrail: 15% max)
GUARDRAIL: BREACHED 

             mean  size
arm                    
control    0.2504  1146
treatment  0.1500  1140

EFFECT: 10.04 point reduction in default

outcome    False  True 
arm                    
control      859    287
treatment    969    171

p = 0.000000
VERDICT: significant

injected effect : 8.0 points
measured effect : 10.04 points
difference      : 2.04 points  (sampling noise)


In [ ]:
# ══════════════════════════════════════════════════════════════
# A/B TEST — change CAP here, run the whole cell
# ══════════════════════════════════════════════════════════════
CAP            = 1.2
TRUE_CONTROL   = 0.236     # observed baseline, both arms
TRUE_TREATMENT = 0.156     # injected effect: 8-point reduction
REQUIRED_N     = 389       # from the step-3 power analysis
GUARDRAIL      = 0.15      # max acceptable book-level volume drop

rng = np.random.default_rng(RANDOM_STATE)


# ── STEP 5a — eligible population + stratified randomisation ──
eligible = loans_train1[loans_train1["inflow_to_loan_ratio"] > CAP].copy()

eligible["ratio_band"] = pd.qcut(eligible["inflow_to_loan_ratio"], 4)
eligible["stratum"]    = (eligible["ratio_band"].astype(str)
                          + " | " + eligible["region"].astype(str))

eligible = eligible.sample(frac=1, random_state=RANDOM_STATE)
eligible["arm"] = np.where(
    eligible.groupby("stratum", observed=True).cumcount() % 2 == 0,
    "control", "treatment"
)

n_arm = eligible["arm"].value_counts().min()
print(f"CAP = {CAP}")
print(f"eligible loans : {len(eligible)}   ({len(eligible)/len(loans_train1):.1%} of the book)")
print(f"n per arm      : {n_arm}   (need {REQUIRED_N})")
print("POWERED        :", "YES" if n_arm >= REQUIRED_N else "NO - underpowered\n")


# ── STEP 5b — balance check ───────────────────────────────────
print("\n--- balance ---")
print(eligible.groupby("arm")[["credit_score", "wallet_tenure_months",
                               "avg_monthly_inflow_pkr", "inflow_to_loan_ratio",
                               "amount_pkr"]].median().round(1))
print("\nbaseline default (should match):")
print(eligible.groupby("arm")["defaulted"].agg(["mean", "size"]).round(4))


# ── STEP 6a — GUARDRAIL, measured against the WHOLE BOOK ──────
book_before = loans_train1["amount_pkr"].sum()
capped_all  = np.where(
    loans_train1["inflow_to_loan_ratio"] > CAP,
    loans_train1["avg_monthly_inflow_pkr"] * CAP,   # above cap → shrunk
    loans_train1["amount_pkr"],                     # below cap → untouched
)
book_after = capped_all.sum()
book_drop  = (book_before - book_after) / book_before

print(f"\n--- guardrail ---")
print(f"book before    : {book_before:>15,.0f}")
print(f"book after     : {book_after:>15,.0f}")
print(f"BOOK-LEVEL DROP: {book_drop:.1%}   (max {GUARDRAIL:.0%})")
print("GUARDRAIL      :", "PASS" if book_drop <= GUARDRAIL else "BREACHED")


# ── STEP 6b — simulate outcomes ───────────────────────────────
p = np.where(eligible["arm"] == "treatment", TRUE_TREATMENT, TRUE_CONTROL)
eligible["outcome"] = rng.random(len(eligible)) < p


# ── STEP 7 — effect size ──────────────────────────────────────
res = eligible.groupby("arm")["outcome"].agg(["mean", "size"])
gap = res.loc["control", "mean"] - res.loc["treatment", "mean"]
print(f"\n--- result ---")
print(res.round(4))
print(f"EFFECT: {gap*100:.2f} point reduction")


# ── STEP 8 — chi-square ───────────────────────────────────────
chi2, pval, dof, expected = chi2_contingency(
    pd.crosstab(eligible["arm"], eligible["outcome"])
)
print(f"p = {pval:.6f}   ->", "significant" if pval < 0.05 else "NOT significant")


# ── STEP 9 — validity check ───────────────────────────────────
injected = (TRUE_CONTROL - TRUE_TREATMENT) * 100
print(f"\ninjected {injected:.1f} pts | measured {gap*100:.2f} pts | "
      f"noise {abs(gap*100 - injected):.2f} pts")

CAP = 1.2
eligible loans : 2597   (40.6% of the book)
n per arm      : 1292   (need 389)
POWERED        : YES

--- balance ---
           credit_score  wallet_tenure_months  avg_monthly_inflow_pkr  \
arm                                                                     
control           466.0                  24.0                 27300.0   
treatment         465.0                  24.0                 25700.0   

           inflow_to_loan_ratio  amount_pkr  
arm                                          
control                     3.4    106752.0  
treatment                   3.4    101276.0  

baseline default (should match):
             mean  size
arm                    
control    0.2360  1305
treatment  0.2353  1292

--- guardrail ---
book before    :     446,003,047
book after     :     170,210,632
BOOK-LEVEL DROP: 61.8%   (max 15%)
GUARDRAIL      : BREACHED

--- result ---
             mean  size
arm                    
control    0.2444  1305
treatment  0.1625  1292
EFFECT: 

In [37]:
# ══════════════════════════════════════════════════════════════
# CAP SWEEP — deterministic, no simulation needed
# ══════════════════════════════════════════════════════════════
book_before = loans_train1["amount_pkr"].sum()
REQUIRED_N  = 389
GUARDRAIL   = 0.15

rows = []
for cap in [1.2, 1.5, 2.0, 2.5, 3.0, 4.0, 5.0, 6.0, 8.0, 10.0]:
    above  = loans_train1["inflow_to_loan_ratio"] > cap
    capped = np.where(above,
                      loans_train1["avg_monthly_inflow_pkr"] * cap,
                      loans_train1["amount_pkr"])
    drop   = (book_before - capped.sum()) / book_before
    n_arm  = above.sum() // 2

    rows.append({
        "cap":          cap,
        "loans":        above.sum(),
        "pct_of_book":  above.mean(),
        "book_drop":    drop,
        "n_per_arm":    n_arm,
        "powered":      "yes" if n_arm >= REQUIRED_N else "NO",
        "guardrail":    "PASS" if drop <= GUARDRAIL else "breach",
    })

sweep = pd.DataFrame(rows)
print(sweep.assign(
    pct_of_book = lambda d: d["pct_of_book"].map("{:.1%}".format),
    book_drop   = lambda d: d["book_drop"].map("{:.1%}".format),
).to_string(index=False))

 cap  loans pct_of_book book_drop  n_per_arm powered guardrail
 1.2   2597       40.6%     61.8%       1298     yes    breach
 1.5   2286       35.8%     56.6%       1143     yes    breach
 2.0   1899       29.7%     49.3%        949     yes    breach
 2.5   1620       25.3%     43.4%        810     yes    breach
 3.0   1421       22.2%     38.5%        710     yes    breach
 4.0   1147       17.9%     30.4%        573     yes    breach
 5.0    927       14.5%     24.3%        463     yes    breach
 6.0    755       11.8%     19.7%        377      NO    breach
 8.0    542        8.5%     13.1%        271      NO      PASS
10.0    381        6.0%      8.8%        190      NO      PASS
